# Consumption profiles with `NewDatabase`

This notebook illustrates the three methods to declare consumption profiles with `shrecc`. 

In [ ]:
from pathlib import Path

import bw2data as bd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from shrecc import NewDatabase
from shrecc.analysis import (
    compare_electricity_generation_mixes,
    compare_electricity_lcia,
)

%load_ext jupyter_black


In [ ]:
# Geographical scope and common 52-week annual period
years = [2025, 2035, 2040]
countries = ["ES", "FR", "DE", "IT", "PT", "BE", "NL", "LU", "AT", "CH"]
time_range = ["2040-01-01 00:00:00", "2040-12-30 23:00:00"]
country_to_compare = "FR"

# Brightway configuration
project_name = "SHRECCei311"
my_db_name = "shrecc_annual_profiles"
write_database = True

climate_method = (
    "ecoinvent-3.11",
    "EF v3.1",
    "climate change",
    "global warming potential (GWP100)",
)

## Create the flat consumption profile

In this paradigm, all hours "are created equal", a low-demand hour has the same weight as a high-demand hour.

In [ ]:
flat_consumption = NewDatabase(
    years=years,
    scenario="DE",
    climate_year=2009,
    countries=countries,
    time_range=time_range,
    project_name=project_name,
    my_db_name=f"{my_db_name}_flat",
    bg_db_name={
        2025: "ecoinvent-3.11-cutoff",
        2035: "ei_cutoff_3.11_remind-eu_SSP2-PkBudg650_2035 2026-07-10",
        2040: "ei_cutoff_3.11_remind-eu_SSP2-PkBudg650_2040 2026-07-10",
    },
    include_consumption_mix_volume=False,
    retain_hourly_results=False,
    consumption_profile="flat",
)

In [ ]:
flat_consumption.create()

## Create the national demand profile

This creates inventories that are representative of the _volumes_ of electricity consumed nationally. This is implicitly what every annual-average dataset is offering, including ecoinvent and premise.

In [ ]:
national_consumption = NewDatabase(
    years=years,
    scenario="DE",
    climate_year=2009,
    countries=countries,
    time_range=time_range,
    project_name=project_name,
    my_db_name=f"{my_db_name}_national_demand",
    bg_db_name={
        2025: "ecoinvent-3.11-cutoff",
        2035: "ei_cutoff_3.11_remind-eu_SSP2-PkBudg650_2035 2026-07-10",
        2040: "ei_cutoff_3.11_remind-eu_SSP2-PkBudg650_2040 2026-07-10",
    },
    include_consumption_mix_volume=False,
    retain_hourly_results=False,
    consumption_profile="national_demand",
)

In [ ]:
national_consumption.create()

## Custom consumption profile

In some LCA studies, you may have access to hourly consumption profiles, which are usually quite different from a flat or national-demand shape (if you study storage technologies, you typically look for load-shifting behaviours).

We can use external data sources to exemplify this, e.g. the ELMAS datasets, developed by MinesParisTech, publicly available [here](https://figshare.com/articles/dataset/ELMAS_dataset/23889780).

In [ ]:
ELMAS_CLUSTER_LABELS = {
    "1": "Manufacturing processes",
    "2": "Trades (non-food)",
    "3": "Trades (food)",
    "4": "Education",
    "5": "Office",
    "6": "Crop farming and transportation",
    "7": "Livestock farming",
    "8": "Water supply and telecommunications",
    "9": "Restaurants",
    "10": "Food industry",
    "11": "Wine industry",
    "12": "Energy supply and rental activities",
    "13": "Hotels",
    "14": "Bakery",
    "15": "Property management companies",
    "16": "Hospital activities",
    "17": "Recreational and social activities",
    "18": "Construction",
}

In [ ]:
profile_path = Path("../data/profiles/ELMAS_dataset") / "Time_series_18_clusters.csv"

elmas_profiles = pd.read_csv(
    profile_path,
    sep=";",
    decimal=",",
    parse_dates=["Time"],
).set_index("Time")

elmas_profiles = elmas_profiles.rename(columns=ELMAS_CLUSTER_LABELS)

In [ ]:
ax = (elmas_profiles / 1_000).plot(figsize=(16, 6))

ax.legend(bbox_to_anchor=(1.05, 0.5), loc="center left")
ax.set_ylabel("Average power (MW)")


Hourly solver results are not retained in this comparison. Keeping only the
profile-weighted activity mixes and database tables prevents several
multi-gigabyte annual arrays from accumulating across the three assumptions.


In [ ]:
# ELMAS contains 52 weeks from 2018. Rebase its calendar to the model year.
target_year = pd.Timestamp(time_range[0]).year
livestock_profile = elmas_profiles["Livestock farming"].copy()
livestock_profile.index += pd.DateOffset(
    years=target_year - livestock_profile.index[0].year
)


In [ ]:
# The custom Series defines the same full 52-week annual scope.
livestock_profile.plot(
    figsize=(14, 3),
    ylabel="Hourly consumption (kWh)",
    title="Annual ELMAS livestock consumption profile",
)


In [ ]:
livestock_profile.describe()


In [ ]:
custom_consumption = NewDatabase(
    years=years,
    scenario="DE",
    climate_year=2009,
    countries=countries,
    project_name=project_name,
    my_db_name=f"{my_db_name}_elmas_livestock",
    bg_db_name={
        2025: "ecoinvent-3.11-cutoff",
        2035: "ei_cutoff_3.11_remind-eu_SSP2-PkBudg650_2035 2026-07-10",
        2040: "ei_cutoff_3.11_remind-eu_SSP2-PkBudg650_2040 2026-07-10",
    },
    include_consumption_mix_volume=False,
    retain_hourly_results=False,
    consumption_profile=livestock_profile,
)

In [ ]:
custom_consumption.create()

## Compare electricity mixes

The three SHRECC inventories and the corresponding background low-voltage
market are resolved to their generating activities. This follows the same
delivered-electricity comparison used in the annual validation notebook.


In [ ]:
profile_models = {
    "Flat": flat_consumption,
    "National demand": national_consumption,
    "ELMAS livestock": custom_consumption,
}
model_order = [*profile_models, "Background"]

bd.projects.set_current(project_name)

missing_backgrounds = {
    name
    for name in flat_consumption.background_databases.values()
    if name not in bd.databases
}
if missing_backgrounds:
    raise ValueError(f"Missing background databases: {missing_backgrounds}")
if climate_method not in bd.methods:
    raise ValueError(f"Brightway method is not installed: {climate_method}")
if not write_database:
    raise RuntimeError(
        "Set write_database=True to compare resolved inventories and calculate LCIA."
    )

for label, model in profile_models.items():
    print(f"Writing {label} inventories")
    model.write()


In [ ]:
comparison_foregrounds = {
    label: model.database_names
    for label, model in profile_models.items()
}
comparison_backgrounds = flat_consumption.background_databases


In [ ]:
mix_comparison, mix_gaps, mix_resolution_diagnostics = (
    compare_electricity_generation_mixes(
        comparison_foregrounds,
        comparison_backgrounds,
        countries=countries,
        years=years,
    )
)

display(
    mix_gaps if not mix_gaps.empty else "No electricity-mix comparison gaps"
)
display(mix_resolution_diagnostics.head())
display(mix_comparison.head())


In [ ]:
def plot_profile_mix_comparison(country, top_n=12):
    country_data = mix_comparison.loc[
        mix_comparison["country"] == country
    ].copy()
    if country_data.empty:
        raise ValueError(f"No electricity-mix comparison data for {country}")

    leading_technologies = (
        country_data.groupby("technology")["share"]
        .sum()
        .nlargest(top_n)
        .index
    )
    country_data["technology_plot"] = country_data["technology"].where(
        country_data["technology"].isin(leading_technologies),
        "REST",
    )
    technologies = list(leading_technologies) + ["REST"]
    colors = dict(
        zip(technologies, plt.cm.tab20(np.linspace(0, 1, len(technologies))))
    )

    fig, axes = plt.subplots(
        len(years),
        1,
        figsize=(13, 3.1 * len(years)),
        sharex=True,
        squeeze=False,
    )
    axes = axes[:, 0]

    for year, ax in zip(years, axes):
        plotted = (
            country_data.loc[country_data["year"] == year]
            .pivot_table(
                index="model",
                columns="technology_plot",
                values="share",
                aggfunc="sum",
                fill_value=0,
            )
            .reindex(
                index=model_order,
                columns=technologies,
                fill_value=0,
            )
        )
        plotted.plot.barh(
            stacked=True,
            ax=ax,
            color=[colors[column] for column in plotted.columns],
            legend=False,
            width=0.72,
        )
        ax.set(ylabel=str(year), xlim=(0, 1))
        ax.grid(axis="x", alpha=0.2)

    handles = [
        plt.Rectangle((0, 0), 1, 1, color=colors[label], label=label)
        for label in technologies
    ]
    fig.legend(
        handles=handles,
        title="Generating activity",
        loc="center left",
        bbox_to_anchor=(0.76, 0.5),
        frameon=False,
    )
    axes[-1].set_xlabel("Electricity share")
    fig.suptitle(f"Electricity composition consumed in {country}")
    fig.subplots_adjust(
        left=0.16,
        right=0.74,
        top=0.93,
        bottom=0.08,
        hspace=0.24,
    )
    return fig, axes


plot_profile_mix_comparison(country_to_compare)


## Compare climate-change impacts

Each score represents 1 kWh of delivered low-voltage electricity. The three
consumption-profile assumptions are compared with the same background market
for each country and model year.


In [ ]:
lcia_results, lcia_gaps = compare_electricity_lcia(
    comparison_foregrounds,
    comparison_backgrounds,
    countries=countries,
    years=years,
    method=climate_method,
    verbose=True,
)
display(lcia_gaps if not lcia_gaps.empty else "No LCIA activity gaps")

lcia_comparison = lcia_results.pivot_table(
    index=["year", "country"],
    columns="model",
    values="score",
).reindex(columns=model_order)

for model in profile_models:
    lcia_comparison[f"{model} difference (%)"] = (
        100
        * (lcia_comparison[model] - lcia_comparison["Background"])
        / lcia_comparison["Background"]
    )

display(lcia_comparison)


In [ ]:
method_unit = bd.Method(climate_method).metadata.get("unit", "kg CO2-eq")

country_lcia = (
    lcia_results.loc[lcia_results["country"] == country_to_compare]
    .pivot(index="year", columns="model", values="score")
    .reindex(index=years, columns=model_order)
)

ax = country_lcia.plot.bar(
    figsize=(10, 4.8),
    color={
        "Flat": "#3B6F8F",
        "National demand": "#D18B35",
        "ELMAS livestock": "#4F8A5B",
        "Background": "#777777",
    },
)
ax.set(
    title=(
        "Climate-change impact of electricity consumed "
        f"in {country_to_compare}"
    ),
    xlabel="Model year",
    ylabel=f"{method_unit} per kWh",
)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
